In [ ]:
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 4 if torch.cuda.is_available() else 0
print(f'Device: {device}, num_workers: {NUM_WORKERS}')

In [ ]:
# Конфиг экспериментов — зеркалит ncf_experiments.ipynb
K_VALUES      = [5, 10, 20]
SEARCH_K      = 10
EVAL_USERS    = 500
SEARCH_EPOCHS = 10
FINAL_EPOCHS  = 20
N_TRIALS      = 40
BATCH_SIZE    = 4096

LAYER_CONFIGS = {
    'small':  [64, 32],
    'medium': [128, 64],
    'large':  [256, 128, 64],
    'xlarge': [256, 128, 64, 32],
}

In [ ]:
df = pd.read_csv('../data/children_products/clildren_product_cleaned.csv')

df_filtered = df[(df['Статус'] == 'Доставлен') & (df['Отменено'] == 'Нет')].copy()
df_filtered = df_filtered.dropna(subset=['Телефон_new', 'ID_SKU', 'Дата'])
df_filtered['Дата'] = pd.to_datetime(df_filtered['Дата'], errors='coerce')
df_filtered = df_filtered.dropna(subset=['Дата'])

MIN_INTERACTIONS = 3
user_counts = df_filtered.groupby('Телефон_new').size()
item_counts = df_filtered.groupby('ID_SKU').size()
df_filtered = df_filtered[
    df_filtered['Телефон_new'].isin(user_counts[user_counts >= MIN_INTERACTIONS].index) &
    df_filtered['ID_SKU'].isin(item_counts[item_counts >= MIN_INTERACTIONS].index)
]

print(f'Пользователей:  {df_filtered["Телефон_new"].nunique():,}')
print(f'Товаров:        {df_filtered["ID_SKU"].nunique():,}')
print(f'Взаимодействий: {len(df_filtered):,}')

In [ ]:
# Уникальные пары пользователь-товар + дата последней покупки
interactions = (
    df_filtered
    .groupby(['Телефон_new', 'ID_SKU'])
    .agg(last_date=('Дата', 'max'), mean_date=('Дата', 'mean'))
    .reset_index()
)

user_enc = LabelEncoder()
item_enc = LabelEncoder()
interactions['user_id'] = user_enc.fit_transform(interactions['Телефон_new'])
interactions['item_id'] = item_enc.fit_transform(interactions['ID_SKU'])

N_USERS = interactions['user_id'].nunique()
N_ITEMS = interactions['item_id'].nunique()
print(f'N_USERS={N_USERS:,}, N_ITEMS={N_ITEMS:,}')

# Train/test split по дате (70/30, зеркалит эксперименты)
split_ts = interactions['last_date'].quantile(0.7)
print(f'Дата разделения: {split_ts}')

train_df = interactions[interactions['last_date'] <  split_ts].copy()
test_df  = interactions[interactions['last_date'] >= split_ts].copy()

# В тесте только пользователи, которые есть в трейне
train_users = set(train_df['user_id'].unique())
test_df = test_df[test_df['user_id'].isin(train_users)]

print(f'Train: {len(train_df):,} пар, {train_df["user_id"].nunique():,} users')
print(f'Test:  {len(test_df):,} пар,  {test_df["user_id"].nunique():,} users')

In [ ]:
def generate_negatives(interactions_df, n_items, n_neg, rng):
    """Генерирует негативные примеры vectorized. Возвращает (users, items) numpy arrays."""
    user_items = interactions_df.groupby('user_id')['item_id'].apply(set).to_dict()
    pos_users  = interactions_df['user_id'].values
    n_pos = len(pos_users)

    # Генерируем кандидатов с запасом (3x), чтобы компенсировать коллизии
    candidates = rng.integers(0, n_items, size=(n_pos, n_neg * 3))

    neg_users, neg_items = [], []
    for i, u in enumerate(pos_users):
        u_set   = user_items.get(int(u), set())
        valid   = [c for c in candidates[i] if c not in u_set][:n_neg]
        # Если кандидатов не хватило — досэмплируем
        while len(valid) < n_neg:
            c = int(rng.integers(0, n_items))
            if c not in u_set:
                valid.append(c)
        neg_users.extend([u] * n_neg)
        neg_items.extend(valid)
    return np.array(neg_users, dtype=np.int64), np.array(neg_items, dtype=np.int64)


class NCFDataset(Dataset):
    """
    Генерирует все негативные примеры один раз при создании.
    Поддерживает веса для временного затухания.
    """
    def __init__(self, interactions_df, n_items, n_neg=4, decay_weights=None, seed=SEED):
        rng = np.random.default_rng(seed)
        t0  = time.time()

        pos_users  = interactions_df['user_id'].values
        pos_items  = interactions_df['item_id'].values
        pos_labels = np.ones(len(pos_users), dtype=np.float32)
        pos_weights = (
            np.asarray(decay_weights, dtype=np.float32)
            if decay_weights is not None
            else np.ones(len(pos_users), dtype=np.float32)
        )

        neg_users, neg_items = generate_negatives(interactions_df, n_items, n_neg, rng)
        neg_labels  = np.zeros(len(neg_users), dtype=np.float32)
        neg_weights = np.ones(len(neg_users),  dtype=np.float32)

        all_users   = np.concatenate([pos_users,   neg_users])
        all_items   = np.concatenate([pos_items,   neg_items])
        all_labels  = np.concatenate([pos_labels,  neg_labels])
        all_weights = np.concatenate([pos_weights, neg_weights])

        # Перемешиваем
        idx = rng.permutation(len(all_users))
        self.users   = torch.from_numpy(all_users[idx])
        self.items   = torch.from_numpy(all_items[idx])
        self.labels  = torch.from_numpy(all_labels[idx])
        self.weights = torch.from_numpy(all_weights[idx])

        print(f'Dataset готов: {len(self):,} примеров ({len(pos_users):,} pos + {len(neg_users):,} neg) за {time.time()-t0:.1f}s')

    def __len__(self):  return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx], self.weights[idx]


def make_loader(dataset, batch_size=BATCH_SIZE, shuffle=True):
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'),
        persistent_workers=(NUM_WORKERS > 0),
    )

In [ ]:
class NeuMF(nn.Module):
    def __init__(self, n_users, n_items, n_factors=64, layer_sizes=None, dropout=0.3):
        super().__init__()
        if layer_sizes is None:
            layer_sizes = [256, 128, 64]

        # GMF ветка
        self.user_emb_gmf = nn.Embedding(n_users, n_factors)
        self.item_emb_gmf = nn.Embedding(n_items, n_factors)

        # MLP ветка
        self.user_emb_mlp = nn.Embedding(n_users, n_factors)
        self.item_emb_mlp = nn.Embedding(n_items, n_factors)

        mlp_layers = []
        in_dim = n_factors * 2
        for out_dim in layer_sizes:
            mlp_layers += [
                nn.Linear(in_dim, out_dim),
                nn.BatchNorm1d(out_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_dim = out_dim
        self.mlp = nn.Sequential(*mlp_layers)

        self.fc_out = nn.Linear(n_factors + layer_sizes[-1], 1)
        self._init_weights()

    def _init_weights(self):
        for emb in [self.user_emb_gmf, self.item_emb_gmf,
                    self.user_emb_mlp, self.item_emb_mlp]:
            nn.init.normal_(emb.weight, std=0.01)
        for m in self.mlp:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
        nn.init.xavier_uniform_(self.fc_out.weight)
        nn.init.zeros_(self.fc_out.bias)

    def forward(self, users, items):
        gmf_out = self.user_emb_gmf(users) * self.item_emb_gmf(items)
        mlp_in  = torch.cat([self.user_emb_mlp(users), self.item_emb_mlp(items)], dim=-1)
        mlp_out = self.mlp(mlp_in)
        x = torch.cat([gmf_out, mlp_out], dim=-1)
        return torch.sigmoid(self.fc_out(x).squeeze(-1))


def count_params(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
def train_model(model, loader, n_epochs, lr=1e-3, verbose=2):
    """Обучает модель. Возвращает список средних loss по эпохам."""
    criterion = nn.BCELoss(reduction='none')
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    model.to(device)
    losses = []

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0
        t0 = time.time()

        for users, items, labels, weights in loader:
            users   = users.to(device, non_blocking=True)
            items   = items.to(device, non_blocking=True)
            labels  = labels.to(device, non_blocking=True)
            weights = weights.to(device, non_blocking=True)

            preds = model(users, items)
            loss  = (criterion(preds, labels) * weights).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg = epoch_loss / len(loader)
        losses.append(avg)
        if verbose > 0 and (epoch % verbose == 0 or epoch == 1 or epoch == n_epochs):
            print(f'Epoch {epoch:>3}/{n_epochs}  loss={avg:.4f}  [{time.time()-t0:.1f}s]')

    return losses

In [ ]:
def precision_at_k(rec, rel, k):
    hits = len(set(rec[:k]) & set(rel))
    return hits / k if k > 0 else 0.0

def recall_at_k(rec, rel, k):
    if not rel: return 0.0
    return len(set(rec[:k]) & set(rel)) / len(set(rel))

def map_at_k(rec, rel, k):
    rel_set = set(rel)
    if not rel_set: return 0.0
    score, hits = 0.0, 0
    for i, item in enumerate(rec[:k]):
        if item in rel_set:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(rel_set), k)

def ndcg_at_k(rec, rel, k):
    rel_set = set(rel)
    if not rel_set: return 0.0
    dcg  = sum(1 / np.log2(i + 2) for i, item in enumerate(rec[:k]) if item in rel_set)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel_set), k)))
    return dcg / idcg if idcg > 0 else 0.0


@torch.no_grad()
def score_all_items(model, user_ids, n_items, user_batch=32):
    """
    Считает скор каждого товара для каждого пользователя.
    Возвращает тензор [n_users, n_items] на CPU.
    Батч по пользователям — чтобы не переполнить GPU память.
    """
    model.eval()
    model.to(device)
    item_t = torch.arange(n_items, device=device)
    all_scores = torch.empty(len(user_ids), n_items)

    for start in range(0, len(user_ids), user_batch):
        batch = user_ids[start:start + user_batch]
        b = len(batch)
        u = torch.tensor(batch, device=device).unsqueeze(1).expand(b, n_items).reshape(-1)
        v = item_t.unsqueeze(0).expand(b, -1).reshape(-1)
        scores = model(u, v).reshape(b, n_items)
        all_scores[start:start + b] = scores.cpu()

    return all_scores


def evaluate_model(model, train_df, test_df, n_items, k_values=K_VALUES,
                   eval_users=None, user_batch=32):
    """
    Быстрая оценка: все скоры считаются батчами на GPU.
    eval_users: список user_id для оценки (None = все тестовые).
    """
    train_hist = train_df.groupby('user_id')['item_id'].apply(set).to_dict()
    test_rel   = test_df.groupby('user_id')['item_id'].apply(list).to_dict()
    train_set  = set(train_df['user_id'].unique())

    if eval_users is None:
        eval_users = [u for u in test_rel if u in train_set]

    t0 = time.time()
    scores = score_all_items(model, eval_users, n_items, user_batch)
    print(f'  Scoring {len(eval_users)} users за {time.time()-t0:.1f}s')

    metrics = {k: {'precision': [], 'recall': [], 'map': [], 'ndcg': []} for k in k_values}
    max_k   = max(k_values)

    for i, u in enumerate(eval_users):
        s = scores[i].clone()
        for train_item in train_hist.get(u, set()):
            if train_item < n_items:
                s[train_item] = -1e9

        _, top_idx = torch.topk(s, min(max_k, n_items))
        rec = top_idx.tolist()
        rel = test_rel.get(u, [])

        for k in k_values:
            metrics[k]['precision'].append(precision_at_k(rec, rel, k))
            metrics[k]['recall'].append(recall_at_k(rec, rel, k))
            metrics[k]['map'].append(map_at_k(rec, rel, k))
            metrics[k]['ndcg'].append(ndcg_at_k(rec, rel, k))

    return {
        k: {m: float(np.mean(v)) for m, v in mv.items()}
        for k, mv in metrics.items()
    }


def quick_ndcg(model, train_df, test_df, n_items, eval_users, k=SEARCH_K):
    res = evaluate_model(model, train_df, test_df, n_items,
                         k_values=[k], eval_users=eval_users)
    return res[k]['ndcg']

In [ ]:
rng_eval = np.random.default_rng(SEED)
test_users_all = [u for u in test_df['user_id'].unique() if u in train_users]
eval_sample = rng_eval.choice(
    test_users_all,
    size=min(EVAL_USERS, len(test_users_all)),
    replace=False,
).tolist()
print(f'Quick-eval sample: {len(eval_sample)} users')

In [ ]:
print('Подготовка baseline dataset (n_neg=4)...')
baseline_dataset = NCFDataset(train_df, N_ITEMS, n_neg=4)
baseline_loader  = make_loader(baseline_dataset)

baseline_model = NeuMF(N_USERS, N_ITEMS, n_factors=64, layer_sizes=[256, 128, 64])
print(f'Параметров: {count_params(baseline_model):,}')

print('\nОбучение baseline NCF (NeuMF, 20 эпох)...')
t0 = time.time()
baseline_losses = train_model(baseline_model, baseline_loader, n_epochs=FINAL_EPOCHS, lr=1e-3, verbose=2)
print(f'Готово за {time.time()-t0:.0f}s')

In [ ]:
print('Оценка baseline...')
baseline_results = evaluate_model(baseline_model, train_df, test_df, N_ITEMS)

print('\n=== Baseline PyTorch NCF ===')
for k in K_VALUES:
    r = baseline_results[k]
    print(f'  K={k}: P={r["precision"]:.4f}  R={r["recall"]:.4f}  MAP={r["map"]:.4f}  NDCG={r["ndcg"]:.4f}')

## Optuna: поиск гиперпараметров

In [ ]:
def make_objective(train_df, test_df, n_users, n_items, eval_users, n_epochs=SEARCH_EPOCHS):
    def objective(trial):
        n_factors  = trial.suggest_int('n_factors', 16, 128)
        lr         = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        layer_key  = trial.suggest_categorical('layers', list(LAYER_CONFIGS.keys()))
        n_neg      = trial.suggest_int('n_neg', 2, 8)
        dropout    = trial.suggest_float('dropout', 0.1, 0.5)

        dataset = NCFDataset(train_df, n_items, n_neg=n_neg)
        loader  = make_loader(dataset)

        model = NeuMF(n_users, n_items,
                      n_factors=n_factors,
                      layer_sizes=LAYER_CONFIGS[layer_key],
                      dropout=dropout)

        train_model(model, loader, n_epochs=n_epochs, lr=lr, verbose=0)
        return quick_ndcg(model, train_df, test_df, n_items, eval_users)

    return objective


print(f'Запуск Optuna: {N_TRIALS} trials, {SEARCH_EPOCHS} эпох каждый...')
t0 = time.time()
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(
    make_objective(train_df, test_df, N_USERS, N_ITEMS, eval_sample),
    n_trials=N_TRIALS,
    show_progress_bar=True,
)
print(f'\nГотово за {time.time()-t0:.0f}s')
print(f'Лучший NDCG@{SEARCH_K}: {study.best_value:.4f}')
print(f'Лучшие параметры: {study.best_params}')

In [ ]:
trials_df = study.trials_dataframe()
top10 = trials_df.sort_values('value', ascending=False).head(10)
display(top10[['number', 'value', 'params_n_factors', 'params_lr',
               'params_layers', 'params_n_neg', 'params_dropout']]
        .rename(columns={'value': f'NDCG@{SEARCH_K}'}))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(trials_df['number'], trials_df['value'], alpha=0.5, linewidth=1)
ax.plot(trials_df['number'], trials_df['value'].cummax(),
        linewidth=2, color='red', label='Best so far')
ax.set_xlabel('Trial'); ax.set_ylabel(f'NDCG@{SEARCH_K}')
ax.set_title('Optuna: прогресс поиска гиперпараметров')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
bp = study.best_params
BEST_N_FACTORS = bp['n_factors']
BEST_LR        = bp['lr']
BEST_LAYERS    = LAYER_CONFIGS[bp['layers']]
BEST_N_NEG     = bp['n_neg']
BEST_DROPOUT   = bp['dropout']
print(f'n_factors={BEST_N_FACTORS}, lr={BEST_LR:.2e}, layers={BEST_LAYERS}, '
      f'n_neg={BEST_N_NEG}, dropout={BEST_DROPOUT:.2f}')

print('\nПодготовка dataset с лучшими параметрами...')
best_dataset = NCFDataset(train_df, N_ITEMS, n_neg=BEST_N_NEG)
best_loader  = make_loader(best_dataset)

best_model = NeuMF(N_USERS, N_ITEMS, n_factors=BEST_N_FACTORS,
                   layer_sizes=BEST_LAYERS, dropout=BEST_DROPOUT)
t0 = time.time()
train_model(best_model, best_loader, n_epochs=FINAL_EPOCHS, lr=BEST_LR, verbose=2)
print(f'Готово за {time.time()-t0:.0f}s')

print('\nОценка...')
best_results = evaluate_model(best_model, train_df, test_df, N_ITEMS)

print(f'\n{"K":>4}  {"Метрика":>12}  {"Baseline":>9}  {"BestHP":>9}  {"Δ":>8}')
print('-' * 52)
for k in K_VALUES:
    for m in ['precision', 'recall', 'map', 'ndcg']:
        base = baseline_results[k][m]
        best = best_results[k][m]
        d = best - base
        print(f'{k:>4}  {m+"@"+str(k):>12}  {base:>9.4f}  {best:>9.4f}  {"+" if d>=0 else ""}{d:>7.4f}')

In [ ]:
# Дни до точки разделения — чем старее покупка, тем меньше вес
days_before_split = (
    split_ts - train_df['mean_date']
).dt.total_seconds().values / 86400
days_before_split = np.clip(days_before_split, 0, None)

DECAY_LAMBDAS = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1]
decay_scores  = []

for lam in DECAY_LAMBDAS:
    t0 = time.time()
    weights = np.exp(-lam * days_before_split).clip(1e-6)

    dataset = NCFDataset(train_df, N_ITEMS, n_neg=BEST_N_NEG, decay_weights=weights)
    loader  = make_loader(dataset)

    model = NeuMF(N_USERS, N_ITEMS, n_factors=BEST_N_FACTORS,
                  layer_sizes=BEST_LAYERS, dropout=BEST_DROPOUT)
    train_model(model, loader, n_epochs=SEARCH_EPOCHS, lr=BEST_LR, verbose=0)

    score = quick_ndcg(model, train_df, test_df, N_ITEMS, eval_sample)
    decay_scores.append(score)
    print(f'  λ={lam:.3f}  NDCG@{SEARCH_K}={score:.4f}  ({time.time()-t0:.0f}s)')

BEST_LAMBDA = DECAY_LAMBDAS[int(np.argmax(decay_scores))]
print(f'\nЛучшая λ = {BEST_LAMBDA}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(DECAY_LAMBDAS, decay_scores, marker='o', linewidth=2)
ax.axvline(BEST_LAMBDA, color='red', linestyle='--', alpha=0.6, label=f'λ*={BEST_LAMBDA}')
ax.set_xscale('log')
ax.set_xlabel('λ (log scale)'); ax.set_ylabel(f'NDCG@{SEARCH_K}')
ax.set_title('Влияние временного затухания на качество NCF')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Финальное обучение с лучшей λ
best_weights = np.exp(-BEST_LAMBDA * days_before_split).clip(1e-6)
decay_dataset = NCFDataset(train_df, N_ITEMS, n_neg=BEST_N_NEG, decay_weights=best_weights)
decay_loader  = make_loader(decay_dataset)

decay_model = NeuMF(N_USERS, N_ITEMS, n_factors=BEST_N_FACTORS,
                    layer_sizes=BEST_LAYERS, dropout=BEST_DROPOUT)
t0 = time.time()
train_model(decay_model, decay_loader, n_epochs=FINAL_EPOCHS, lr=BEST_LR, verbose=2)
print(f'Готово за {time.time()-t0:.0f}s')

print('\nОценка...')
decay_results = evaluate_model(decay_model, train_df, test_df, N_ITEMS)

In [ ]:
all_variants = {
    'baseline_pt':     baseline_results,
    'best_hparams_pt': best_results,
    'time_decay_pt':   decay_results,
}

ref = baseline_results
print(f'{"Вариант":>22}  {"NDCG@10":>9}  {"MAP@10":>9}  {"ΔNDCG@10":>10}')
print('-' * 60)
for name, res in all_variants.items():
    ndcg = res[10]['ndcg']
    mmap = res[10]['map']
    d    = ndcg - ref[10]['ndcg']
    print(f'{name:>22}  {ndcg:>9.4f}  {mmap:>9.4f}  {"+" if d>=0 else ""}{d:>9.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = sns.color_palette('tab10', len(all_variants))

for ax, metric in zip(axes, ['ndcg', 'map']):
    for (name, res), color in zip(all_variants.items(), colors):
        y = [res[k][metric] for k in K_VALUES]
        ax.plot(K_VALUES, y, marker='o', linewidth=1.5, label=name, color=color)
    ax.set_xlabel('K'); ax.set_ylabel(f'{metric.upper()}@K')
    ax.set_title(f'{metric.upper()}@K')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle('NCF Experiments — PyTorch', fontsize=12)
plt.tight_layout()
plt.show()